In [0]:
%sql

create schema if not exists ammu_catalog.bronze;

In [0]:
source_path = "abfss://external-location@amrutacodesstorage.dfs.core.windows.net/external/hospital/"
checkpoint_path = "abfss://external-location@amrutacodesstorage.dfs.core.windows.net/bronze/hospitals_raw/checkpoint/"
schema_location = "abfss://external-location@amrutacodesstorage.dfs.core.windows.net/bronze/hospitals_raw/schema/"

# Autoloader read
df = (spark.readStream
          .format("cloudFiles")
          .option("cloudFiles.format", "csv")
          .option("header", "true")
          .option("inferSchema", "true")
          .option("cloudFiles.maxFilesPerTrigger", 1) # READ ONE FILE AT A TIME
          .option("cloudFiles.schemaLocation", schema_location)  
          .load(source_path)
     )

# Write Bronze table (append)
( df.drop("_rescued_data")
      .writeStream
      .format("delta")
      .option("checkpointLocation", checkpoint_path)
      .option(
    "path",
    "abfss://external-location@amrutacodesstorage.dfs.core.windows.net/bronze/hospitals_raw"
)
      .outputMode("append")
      .trigger(availableNow=True)
      .start()
)

In [0]:
%sql

CREATE TABLE IF NOT EXISTS ammu_catalog.bronze.hospitals_raw
USING DELTA
LOCATION 'abfss://extrenal-location@amrutacodesstorage.dfs.core.windows.net/bronze/hospitals_raw';

In [0]:
%sql
select * from ammu_catalog.bronze.hospitals_raw

In [0]:
%sql
LIST 'abfss://extrenal-location@amrutacodesstorage.dfs.core.windows.net/bronze/hospitals_raw/';

In [0]:
display(dbutils.fs.ls(source_path))

In [0]:
source_path = "abfss://extrenal-location@amrutacodesstorage.dfs.core.windows.net/external/hospital/"
checkpoint_path = "abfss://extrenal-location@amrutacodesstorage.dfs.core.windows.net/bronze/hospital_raw/checkpoint/"
schema_location = "abfss://extrenal-location@amrutacodesstorage.dfs.core.windows.net/bronze/hospital_raw/schema/"

df = (spark.readStream
          .format("cloudFiles")
          .option("cloudFiles.format", "csv")
          .option("header", "true")
          .option("inferSchema", "true")
          .option("cloudFiles.maxFilesPerTrigger", 1) # READ ONE FILE AT A TIME
          .option("cloudFiles.schemaLocation", schema_location)  
          .load(source_path)
     )

# Write Bronze table (append)
(
    df.drop("_rescued_data")
      .writeStream
      .format("delta")
      .option("checkpointLocation", checkpoint_path)
      .option(
          "path",
          "abfss://extrenal-location@amrutacodesstorage.dfs.core.windows.net/bronze/hospital_raw"
      )
      .outputMode("append")
      .trigger(availableNow=True)
      .start()
)

In [0]:
%sql
CREATE TABLE IF NOT EXISTS ammu_catalog.bronze.hospital_raw
USING DELTA
LOCATION 'abfss://extrenal-location@amrutacodesstorage.dfs.core.windows.net/bronze/hospital_raw';

In [0]:
%sql
select * from ammu_catalog.bronze.hospital_raw